# Priprema podataka

In [1]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0




  0%|          | 0.00/41.0M [00:00<?, ?B/s]
  2%|2         | 1.00M/41.0M [00:00<00:30, 1.39MB/s]
  5%|4         | 2.00M/41.0M [00:00<00:14, 2.74MB/s]
  7%|7         | 3.00M/41.0M [00:00<00:09, 4.12MB/s]
 10%|9         | 4.00M/41.0M [00:01<00:07, 4.94MB/s]
 12%|#2        | 5.00M/41.0M [00:01<00:06, 5.70MB/s]
 15%|#4        | 6.00M/41.0M [00:01<00:06, 5.81MB/s]
 17%|#7        | 7.00M/41.0M [00:01<00:06, 5.75MB/s]
 20%|#9        | 8.00M/41.0M [00:01<00:06, 5.58MB/s]
 22%|##1       | 9.00M/41.0M [00:01<00:05, 6.13MB/s]
 24%|##4       | 10.0M/41.0M [00:02<00:05, 6.25MB/s]
 27%|##6       | 11.0M/41.0M [00:02<00:05, 6.07MB/s]
 29%|##9       | 12.0M/41.0M [00:02<00:05, 5.83MB/s]
 32%|###1      | 13.0M/41.0M [00:02<00:05, 5.71MB/s]
 34%|###4      | 14.0M/41.0M [00:02<00:04, 6.26MB/s]
 37%|###6      | 15.0M/41.0M [00:02<00:04, 6.32MB/s]
 39%|###9      | 16.0M/41.0M [00:03<00:04, 5.89MB/s]
 41%|####1     | 17.0M/41.0M [00:03<00:04, 5.51MB/s]
 44%|####3     | 18.0M/41.0M [00:03<00:04, 5.30MB/s]
 

## Ucitavanje

In [ ]:
import re

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

ps = PorterStemmer()

In [3]:
fake = pd.read_csv("./data/Fake.csv")
fake["label"] = 1
true = pd.read_csv("./data/True.csv")
true["label"] = 0

df = pd.concat([fake, true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(len(df))
print(df["label"].value_counts())

44898
label
1    23481
0    21417
Name: count, dtype: int64


## Funkcije

In [4]:
def ocisti(tekst):
    tekst = str(tekst)

    tekst = re.sub(r"^\s*[A-Za-z .,'/-]{0,60}\(Reuters\)\s*-\s*", "", tekst)

    tekst = re.sub(r"\(reuters\)", " ", tekst, flags=re.IGNORECASE)
    tekst = re.sub(r"\breuters\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(featured image via|image via|featured image|getty images|pic\.twitter\.com|screen capture|screenshot via|photo by)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(https?://\S+|www\.\S+|\b\S+\.com\b)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"@\w+", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\s+", " ", tekst)
    tekst = tekst.strip()

    return tekst


def stemuj(tekst):
    tekst = re.sub("[^a-zA-Z]", " ", str(tekst))
    words = tekst.lower().split()
    words = [ps.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

## Obrada

Prvo se uklanjaju tragovi izvora, pa tek onda stemming.
Obrnut redosled ne radi jer stemmer pretvori reuters u reuter,
january u januari, pa ih obrasci vise ne pogadjaju.

In [5]:
print("Ciscenje teksta, traje nekoliko minuta...")
df["text_ok"] = df["text"].apply(ocisti)

pre_dedup = len(df)
df = df.drop_duplicates(subset="text_ok")
df = df.reset_index(drop=True)
print("izbaceno duplikata:", pre_dedup - len(df))

df["duzina"] = df["text_ok"].str.len()
df = df[df["duzina"] >= 40]
df = df.reset_index(drop=True)

print("Stemming, traje nekoliko minuta...")
df["text_clean"] = df["text_ok"].apply(stemuj)

print()
print("ostalo clanaka:", len(df))
print("jos ima 'reuters':", df["text_ok"].str.contains("reuters", case=False).sum())
print("duplikata po text_ok:", df["text_ok"].duplicated().sum())
print(df["label"].value_counts())

Ciscenje teksta, traje nekoliko minuta...


izbaceno duplikata: 6312


Stemming, traje nekoliko minuta...



ostalo clanaka: 38514


jos ima 'reuters': 16


duplikata po text_ok: 0
label
0    21189
1    17325
Name: count, dtype: int64


## Snimanje

In [ ]:
df = df[["title", "text", "subject", "date", "label", "text_ok", "text_clean", "duzina"]]

df.to_pickle("./data/cleaned_news_data.pkl")

df[["text_ok", "label"]].to_parquet("./data/news_colab.parquet", index=False)

import os
for f in ("cleaned_news_data.pkl", "news_colab.parquet"):
    print(f, round(os.path.getsize("./data/" + f) / 1e6, 1), "MB")